<a href="https://colab.research.google.com/github/monteiro-sara/algorithmic_trading/blob/main/Technical_stock_trading_indicators_Yahoo_Live_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Technical Stock Indicators — Yahoo Finance Live + Interpreter + Backtesting

This version develops the original indicator notebook into a small quantitative-research dashboard.

It contains:

- current Yahoo Finance OHLCV candles
- candlestick / volume dashboard
- EMA / SMA / RSI / MACD / stochastic
- ATR and Bollinger Bands
- a transparent **indicator interpreter**
- manual data refresh
- a corrected Yahoo Finance **WebSocket live stream**
- vectorized BUY / NEUTRAL / SELL signal rules
- no-look-ahead backtesting with transaction costs
- strategy metrics
- benchmark comparison
- multi-ticker scanning
- walk-forward machine-learning evaluation

> This notebook is for research and education. Indicator labels and model outputs are not financial advice and should not be interpreted as guaranteed forecasts.

In [1]:
# Colab setup
!pip -q install "yfinance==1.5.2" plotly ipywidgets nest_asyncio scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.1/144.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 74.8 MB/s eta 0:00:00


In [2]:
import asyncio
import warnings

import numpy as np
import pandas as pd
import yfinance as yf

import plotly.graph_objects as go
from plotly.subplots import make_subplots

import ipywidgets as widgets
from IPython.display import display, Markdown

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    balanced_accuracy_score,
)

import nest_asyncio
nest_asyncio.apply()

warnings.filterwarnings("ignore", category=FutureWarning)

print("yfinance:", yf.__version__)

yfinance: 1.5.2


## 1. Configuration

Change only this block for ordinary use.

Examples of Yahoo symbols:

- `SPY` — S&P 500 ETF
- `QQQ` — Nasdaq-100 ETF
- `AAPL` — Apple
- `NVDA` — NVIDIA
- `BTC-USD` — Bitcoin / USD
- `EURUSD=X` — EUR / USD

In [3]:
# ---------- MARKET DATA ----------
TICKER = "SPY"
PERIOD = "5d"
INTERVAL = "1m"
PREPOST = False
DISPLAY_BARS = 600

# ---------- RESEARCH SETTINGS ----------
BENCHMARK_TICKER = "SPY"
TRANSACTION_COST_BPS = 2.0
ALLOW_SHORTS = False

# Number of bars ahead for the optional ML target.
PREDICTION_HORIZON = 5

# Rule-score thresholds used by the vectorized signal engine.
BUY_THRESHOLD = 2.25
SELL_THRESHOLD = -2.25

## 2. Download Yahoo Finance data

In [4]:
def fetch_yahoo_data(
    ticker=None,
    period=None,
    interval=None,
    prepost=None,
):
    ticker = TICKER if ticker is None else ticker
    period = PERIOD if period is None else period
    interval = INTERVAL if interval is None else interval
    prepost = PREPOST if prepost is None else prepost

    df = yf.download(
        ticker,
        period=period,
        interval=interval,
        auto_adjust=True,
        repair=True,
        prepost=prepost,
        progress=False,
        multi_level_index=False,
        threads=False,
    )

    if df is None or df.empty:
        raise RuntimeError(
            f"No Yahoo Finance data returned for {ticker!r}. "
            "Check the ticker, period, interval, market session, "
            "or Yahoo availability."
        )

    df = df.rename_axis("Datetime").reset_index()

    if "Datetime" not in df.columns and "Date" in df.columns:
        df = df.rename(columns={"Date": "Datetime"})

    wanted = ["Datetime", "Open", "High", "Low", "Close", "Volume"]
    missing = [c for c in wanted if c not in df.columns]
    if missing:
        raise RuntimeError(f"Missing expected Yahoo columns: {missing}")

    df = df[wanted].copy()
    df["Datetime"] = pd.to_datetime(df["Datetime"])
    df = df.dropna(subset=["Open", "High", "Low", "Close"])
    df = df.sort_values("Datetime").drop_duplicates("Datetime").reset_index(drop=True)

    return df


raw_df = fetch_yahoo_data()
print(f"{TICKER}: {len(raw_df):,} bars")
print("First:", raw_df["Datetime"].iloc[0])
print("Latest:", raw_df["Datetime"].iloc[-1])
raw_df.tail()

SPY: 1,950 bars
First: 2026-08-17 09:30:00-04:00
Latest: 2026-08-21 15:59:00-04:00


,Datetime,Open,High,Low,Close,Volume
1945,2026-08-21 15:55:00-04:00,765.890015,765.909973,765.530029,765.849976,331280
1946,2026-08-21 15:56:00-04:00,765.844971,766.020020,765.770020,765.984985,176490
1947,2026-08-21 15:57:00-04:00,765.989990,766.039978,765.869995,765.979980,383020
1948,2026-08-21 15:58:00-04:00,765.969971,766.169983,765.960022,766.104980,512201
1949,2026-08-21 15:59:00-04:00,766.099976,766.234985,765.340027,765.650024,1235702


## 3. Technical indicators

Windows are measured in **bars**.

With `INTERVAL="1m"`, `SMA_200` is a 200-minute-bar average.  
With `INTERVAL="1d"`, it is a 200-trading-day average.

In [5]:
def add_indicators(df):
    out = df.copy()

    close = out["Close"].astype(float)
    high = out["High"].astype(float)
    low = out["Low"].astype(float)
    volume = out["Volume"].astype(float)

    # ---------------- Moving averages ----------------
    out["EMA_9"] = close.ewm(span=9, adjust=False).mean()
    out["EMA_20"] = close.ewm(span=20, adjust=False).mean()
    out["SMA_50"] = close.rolling(50).mean()
    out["SMA_100"] = close.rolling(100).mean()
    out["SMA_200"] = close.rolling(200).mean()

    # ---------------- RSI (Wilder-style) ----------------
    delta = close.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.ewm(alpha=1/14, adjust=False, min_periods=14).mean()
    avg_loss = loss.ewm(alpha=1/14, adjust=False, min_periods=14).mean()

    rs = avg_gain / avg_loss.replace(0, np.nan)
    out["RSI_14"] = 100 - (100 / (1 + rs))

    # ---------------- MACD ----------------
    ema_12 = close.ewm(span=12, adjust=False).mean()
    ema_26 = close.ewm(span=26, adjust=False).mean()

    out["MACD"] = ema_12 - ema_26
    out["MACD_SIGNAL"] = out["MACD"].ewm(span=9, adjust=False).mean()
    out["MACD_HIST"] = out["MACD"] - out["MACD_SIGNAL"]

    # ---------------- Stochastic ----------------
    low_14 = low.rolling(14).min()
    high_14 = high.rolling(14).max()
    stoch_range = (high_14 - low_14).replace(0, np.nan)

    out["STOCH_K"] = 100 * (close - low_14) / stoch_range
    out["STOCH_D"] = out["STOCH_K"].rolling(3).mean()

    # ---------------- ATR ----------------
    prev_close = close.shift(1)
    true_range = pd.concat(
        [
            high - low,
            (high - prev_close).abs(),
            (low - prev_close).abs(),
        ],
        axis=1,
    ).max(axis=1)

    out["ATR_14"] = true_range.ewm(alpha=1/14, adjust=False, min_periods=14).mean()
    out["ATR_PCT"] = 100 * out["ATR_14"] / close

    # ---------------- Bollinger Bands ----------------
    out["BB_MID"] = close.rolling(20).mean()
    bb_std = close.rolling(20).std()
    out["BB_UPPER"] = out["BB_MID"] + 2 * bb_std
    out["BB_LOWER"] = out["BB_MID"] - 2 * bb_std
    out["BB_WIDTH_PCT"] = 100 * (out["BB_UPPER"] - out["BB_LOWER"]) / out["BB_MID"]

    bb_den = (out["BB_UPPER"] - out["BB_LOWER"]).replace(0, np.nan)
    out["BB_POSITION"] = (close - out["BB_LOWER"]) / bb_den

    # ---------------- Volume context ----------------
    out["VOLUME_MA20"] = volume.rolling(20).mean()
    out["VOLUME_RATIO"] = volume / out["VOLUME_MA20"].replace(0, np.nan)

    # ---------------- Returns ----------------
    out["RETURN_1"] = close.pct_change()
    out["RETURN_5"] = close.pct_change(5)
    out["RETURN_20"] = close.pct_change(20)

    return out


df = add_indicators(raw_df)
df.tail()

,Datetime,Open,High,Low,Close,Volume,EMA_9,EMA_20,SMA_50,SMA_100,...,BB_MID,BB_UPPER,BB_LOWER,BB_WIDTH_PCT,BB_POSITION,VOLUME_MA20,VOLUME_RATIO,RETURN_1,RETURN_5,RETURN_20
1945,2026-08-21 15:55:00-04:00,765.890015,765.909973,765.530029,765.849976,331280,766.116230,766.196986,766.152091,765.857473,...,766.270245,766.757506,765.782985,0.127177,0.068742,161822.35,2.047183,-0.000065,-0.000463,-0.000379
1946,2026-08-21 15:56:00-04:00,765.844971,766.020020,765.770020,765.984985,176490,766.089981,766.176796,766.150791,765.863723,...,766.261496,766.763165,765.759827,0.130939,0.224409,165206.10,1.068302,0.000176,-0.000241,-0.000228
1947,2026-08-21 15:57:00-04:00,765.989990,766.039978,765.869995,765.979980,383020,766.067981,766.158051,766.145791,765.869973,...,766.246994,766.764155,765.729833,0.134985,0.241847,181430.55,2.111111,-0.000007,-0.000065,-0.000379
1948,2026-08-21 15:58:00-04:00,765.969971,766.169983,765.960022,766.104980,512201,766.075381,766.152997,766.143491,765.877573,...,766.238742,766.759609,765.717875,0.135954,0.371597,203861.55,2.512494,0.000163,0.000137,-0.000215
1949,2026-08-21 15:59:00-04:00,766.099976,766.234985,765.340027,765.650024,1235702,765.990310,766.105095,766.134492,765.879373,...,766.207492,766.790484,765.624500,0.152176,0.021891,261616.15,4.723340,-0.000594,-0.000326,-0.000816


## 4. Combined technical dashboard

In [6]:
def technical_dashboard(df, ticker=None, display_bars=None):
    ticker = TICKER if ticker is None else ticker
    display_bars = DISPLAY_BARS if display_bars is None else display_bars
    data = df.tail(display_bars).copy() if display_bars else df.copy()

    fig = make_subplots(
        rows=6,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.022,
        row_heights=[0.40, 0.10, 0.13, 0.13, 0.12, 0.12],
        subplot_titles=(
            f"{ticker} — Price, Moving Averages & Bollinger Bands",
            "Volume",
            "RSI (14)",
            "MACD",
            "Stochastic (14, 3)",
            "ATR % of price",
        ),
    )

    # Price
    fig.add_trace(
        go.Candlestick(
            x=data["Datetime"],
            open=data["Open"],
            high=data["High"],
            low=data["Low"],
            close=data["Close"],
            name="OHLC",
        ),
        row=1, col=1,
    )

    for col, name in [
        ("EMA_9", "EMA 9"),
        ("SMA_50", "SMA 50"),
        ("SMA_200", "SMA 200"),
        ("BB_UPPER", "BB upper"),
        ("BB_MID", "BB mid"),
        ("BB_LOWER", "BB lower"),
    ]:
        fig.add_trace(
            go.Scatter(x=data["Datetime"], y=data[col], name=name),
            row=1, col=1,
        )

    # Volume
    fig.add_trace(
        go.Bar(x=data["Datetime"], y=data["Volume"], name="Volume"),
        row=2, col=1,
    )

    # RSI
    fig.add_trace(
        go.Scatter(x=data["Datetime"], y=data["RSI_14"], name="RSI 14"),
        row=3, col=1,
    )
    fig.add_hline(y=70, line_dash="dash", row=3, col=1)
    fig.add_hline(y=50, line_dash="dot", row=3, col=1)
    fig.add_hline(y=30, line_dash="dash", row=3, col=1)

    # MACD
    fig.add_trace(
        go.Scatter(x=data["Datetime"], y=data["MACD"], name="MACD"),
        row=4, col=1,
    )
    fig.add_trace(
        go.Scatter(x=data["Datetime"], y=data["MACD_SIGNAL"], name="MACD signal"),
        row=4, col=1,
    )
    fig.add_trace(
        go.Bar(x=data["Datetime"], y=data["MACD_HIST"], name="MACD hist"),
        row=4, col=1,
    )

    # Stochastic
    fig.add_trace(
        go.Scatter(x=data["Datetime"], y=data["STOCH_K"], name="%K"),
        row=5, col=1,
    )
    fig.add_trace(
        go.Scatter(x=data["Datetime"], y=data["STOCH_D"], name="%D"),
        row=5, col=1,
    )
    fig.add_hline(y=80, line_dash="dash", row=5, col=1)
    fig.add_hline(y=20, line_dash="dash", row=5, col=1)

    # ATR %
    fig.add_trace(
        go.Scatter(x=data["Datetime"], y=data["ATR_PCT"], name="ATR %"),
        row=6, col=1,
    )

    fig.update_layout(
        height=1350,
        title=f"{ticker} | Yahoo Finance | {PERIOD} @ {INTERVAL}",
        xaxis_rangeslider_visible=False,
        hovermode="x unified",
        legend=dict(orientation="h"),
    )

    fig.show()


technical_dashboard(df)

## 5. Indicator interpreter

This interpreter is **rule-based and auditable**.

It does not claim to know the future. It summarizes whether the currently observed indicators agree on:

- trend
- momentum
- MACD direction
- stochastic direction
- volume confirmation
- stretched / overbought / oversold conditions
- volatility

The reported **agreement** is simply agreement among the active rules. It is **not a statistical probability of profit**.

In [7]:
def interpret_market_state(df, ticker=None):
    ticker = TICKER if ticker is None else ticker

    if df.empty:
        raise ValueError("DataFrame is empty.")

    x = df.iloc[-1]
    evidence = []
    cautions = []
    directional_votes = []

    score = 0.0
    possible = 0.0

    def add_vote(points, max_points, text):
        nonlocal score, possible
        if points is None:
            return
        score += points
        possible += max_points
        if points > 0:
            directional_votes.append(1)
        elif points < 0:
            directional_votes.append(-1)
        evidence.append(text)

    close = float(x["Close"])

    # -------- Trend --------
    if pd.notna(x["SMA_50"]):
        if close > x["SMA_50"]:
            add_vote(+1.0, 1.0, "Price is above SMA50 → positive medium-term trend.")
        else:
            add_vote(-1.0, 1.0, "Price is below SMA50 → negative medium-term trend.")

    if pd.notna(x["SMA_200"]):
        if close > x["SMA_200"]:
            add_vote(+1.0, 1.0, "Price is above SMA200 → positive long-term trend context.")
        else:
            add_vote(-1.0, 1.0, "Price is below SMA200 → negative long-term trend context.")

    if pd.notna(x["SMA_50"]) and pd.notna(x["SMA_200"]):
        if x["SMA_50"] > x["SMA_200"]:
            add_vote(+0.75, 0.75, "SMA50 is above SMA200 → bullish moving-average structure.")
        else:
            add_vote(-0.75, 0.75, "SMA50 is below SMA200 → bearish moving-average structure.")

    if pd.notna(x["EMA_9"]) and pd.notna(x["SMA_50"]):
        if x["EMA_9"] > x["SMA_50"]:
            add_vote(+0.50, 0.50, "EMA9 is above SMA50 → short-term trend supports upside.")
        else:
            add_vote(-0.50, 0.50, "EMA9 is below SMA50 → short-term trend supports downside.")

    # -------- RSI --------
    rsi = x["RSI_14"]
    if pd.notna(rsi):
        if rsi >= 70:
            add_vote(+0.35, 1.0, f"RSI={rsi:.1f} → strong upside momentum, but stretched.")
            cautions.append("RSI is >=70: momentum is strong but the market is overbought/stretched.")
        elif rsi >= 55:
            add_vote(+1.0, 1.0, f"RSI={rsi:.1f} → positive momentum.")
        elif rsi > 45:
            evidence.append(f"RSI={rsi:.1f} → neutral momentum zone.")
            possible += 1.0
        elif rsi > 30:
            add_vote(-1.0, 1.0, f"RSI={rsi:.1f} → negative momentum.")
        else:
            add_vote(-0.35, 1.0, f"RSI={rsi:.1f} → strong downside momentum, but stretched.")
            cautions.append("RSI is <=30: downside momentum is strong but the market is oversold/stretched.")

    # -------- MACD --------
    if pd.notna(x["MACD"]) and pd.notna(x["MACD_SIGNAL"]):
        macd_points = 0.0

        macd_points += 0.65 if x["MACD"] > x["MACD_SIGNAL"] else -0.65
        macd_points += 0.35 if x["MACD"] > 0 else -0.35

        direction = "bullish" if macd_points > 0 else "bearish"
        add_vote(
            macd_points,
            1.0,
            f"MACD structure is {direction}: MACD={x['MACD']:.4f}, "
            f"signal={x['MACD_SIGNAL']:.4f}.",
        )

    # -------- Stochastic --------
    k = x["STOCH_K"]
    d = x["STOCH_D"]

    if pd.notna(k) and pd.notna(d):
        if k > d:
            add_vote(+0.60, 0.60, f"Stochastic %K ({k:.1f}) is above %D ({d:.1f}).")
        elif k < d:
            add_vote(-0.60, 0.60, f"Stochastic %K ({k:.1f}) is below %D ({d:.1f}).")
        else:
            possible += 0.60
            evidence.append("Stochastic %K and %D are equal.")

        if k >= 80:
            cautions.append("Stochastic is >=80: short-term price action is stretched upward.")
        elif k <= 20:
            cautions.append("Stochastic is <=20: short-term price action is stretched downward.")

    # -------- Volume confirmation --------
    vr = x["VOLUME_RATIO"]
    if pd.notna(vr):
        evidence.append(f"Current volume is {vr:.2f}× its 20-bar average.")
        if vr >= 1.5:
            if score > 0:
                score += 0.35
                possible += 0.35
                directional_votes.append(1)
                evidence.append("High relative volume confirms the current positive direction.")
            elif score < 0:
                score -= 0.35
                possible += 0.35
                directional_votes.append(-1)
                evidence.append("High relative volume confirms the current negative direction.")

    # -------- Bollinger stretch --------
    bb_pos = x["BB_POSITION"]
    if pd.notna(bb_pos):
        if bb_pos > 1:
            cautions.append("Price is above the upper Bollinger Band.")
        elif bb_pos < 0:
            cautions.append("Price is below the lower Bollinger Band.")

    # -------- Volatility --------
    atr_pct = x["ATR_PCT"]
    if pd.notna(atr_pct):
        evidence.append(f"ATR(14) is {atr_pct:.3f}% of price per bar.")

    normalized = 0.0 if possible == 0 else 100 * score / possible
    normalized = float(np.clip(normalized, -100, 100))

    if normalized >= 55:
        label = "STRONG BULLISH"
    elif normalized >= 20:
        label = "BULLISH"
    elif normalized <= -55:
        label = "STRONG BEARISH"
    elif normalized <= -20:
        label = "BEARISH"
    else:
        label = "NEUTRAL / MIXED"

    if directional_votes:
        majority = 1 if sum(directional_votes) > 0 else -1 if sum(directional_votes) < 0 else 0
        if majority == 0:
            agreement = 50.0
        else:
            agree = sum(v == majority for v in directional_votes)
            agreement = 100 * agree / len(directional_votes)
    else:
        agreement = np.nan

    result = {
        "ticker": ticker,
        "timestamp": x["Datetime"],
        "close": close,
        "label": label,
        "score": normalized,
        "rule_agreement_pct": agreement,
        "evidence": evidence,
        "cautions": cautions,
    }
    return result


def print_interpretation(result):
    agreement = result["rule_agreement_pct"]
    agreement_txt = "n/a" if pd.isna(agreement) else f"{agreement:.0f}%"

    print("=" * 78)
    print(f"{result['ticker']} | {result['label']}")
    print(f"Close: {result['close']:.4f}")
    print(f"Composite rule score: {result['score']:+.1f}/100")
    print(f"Directional rule agreement: {agreement_txt}")
    print(f"Latest bar: {result['timestamp']}")
    print("=" * 78)

    print("\nWHY:")
    for item in result["evidence"]:
        print(" •", item)

    if result["cautions"]:
        print("\nCAUTIONS:")
        for item in result["cautions"]:
            print(" •", item)

    print("\nInterpretation is descriptive, not a probability of future return.")


interpretation = interpret_market_state(df)
print_interpretation(interpretation)

SPY | STRONG BEARISH
Close: 765.6500
Composite rule score: -75.8/100
Directional rule agreement: 88%
Latest bar: 2026-08-21 15:59:00-04:00

WHY:
 • Price is below SMA50 → negative medium-term trend.
 • Price is below SMA200 → negative long-term trend context.
 • SMA50 is above SMA200 → bullish moving-average structure.
 • EMA9 is below SMA50 → short-term trend supports downside.
 • RSI=34.4 → negative momentum.
 • MACD structure is bearish: MACD=-0.0848, signal=-0.0201.
 • Stochastic %K (21.2) is below %D (34.0).
 • Current volume is 4.72× its 20-bar average.
 • High relative volume confirms the current negative direction.
 • ATR(14) is 0.033% of price per bar.

Interpretation is descriptive, not a probability of future return.


## 6. Refresh current Yahoo candles + interpreter

This updates the candles, recalculates the indicators, redraws the dashboard, and reruns the interpreter.

In [8]:
refresh_button = widgets.Button(
    description="Refresh Yahoo data",
    button_style="primary",
    icon="refresh",
)

refresh_output = widgets.Output()

def on_refresh(_):
    global raw_df, df, interpretation

    with refresh_output:
        refresh_output.clear_output(wait=True)

        try:
            raw_df = fetch_yahoo_data()
            df = add_indicators(raw_df)
            interpretation = interpret_market_state(df)

            print_interpretation(interpretation)
            technical_dashboard(df)

        except Exception as exc:
            print(type(exc).__name__ + ":", exc)

refresh_button.on_click(on_refresh)

display(refresh_button, refresh_output)

Button(button_style='primary', description='Refresh Yahoo data', icon='refresh', style=ButtonStyle())

Output()

## 7. Yahoo live WebSocket — corrected

The previous version looked as if it did nothing because it **defined** the stream function but left the line that actually started it commented out.

This version:

1. connects visibly,
2. subscribes visibly,
3. prints incoming messages in a compact format,
4. prints a heartbeat while waiting,
5. stops after a chosen duration,
6. reports clearly if no price messages arrived.

A quiet stream does not necessarily mean the code is broken. A security can have few/no updates outside an active trading session. `BTC-USD` is useful as a 24/7 stream test.

In [9]:
LIVE_SYMBOLS = [TICKER]
STREAM_SECONDS = 30
HEARTBEAT_SECONDS = 5

live_messages = []


def compact_live_message(message):
    symbol = message.get("id", "?")
    price = message.get("price", np.nan)
    event_time = message.get("time", None)
    market_hours = message.get("market_hours", message.get("marketHours", ""))

    if event_time is not None:
        try:
            # Yahoo protobuf time is commonly milliseconds since epoch.
            event_time = pd.to_datetime(int(event_time), unit="ms", utc=True)
        except Exception:
            pass

    if isinstance(price, (int, float)):
        price_txt = f"{price:.6f}"
    else:
        price_txt = str(price)

    return f"{symbol:12s} price={price_txt:>14s} time={event_time} {market_hours}"


def live_message_handler(message):
    live_messages.append(message)
    print("LIVE |", compact_live_message(message))


async def stream_yahoo_live(
    symbols=None,
    seconds=STREAM_SECONDS,
    heartbeat_seconds=HEARTBEAT_SECONDS,
):
    symbols = LIVE_SYMBOLS if symbols is None else symbols
    live_messages.clear()

    print(f"Connecting to Yahoo WebSocket...")
    print(f"Symbols: {symbols}")
    print(f"Listening for {seconds} seconds...\n")

    ws = yf.AsyncWebSocket(verbose=True)
    listener = None

    try:
        await ws.subscribe(symbols)
        listener = asyncio.create_task(ws.listen(live_message_handler))

        elapsed = 0
        while elapsed < seconds:
            step = min(heartbeat_seconds, seconds - elapsed)
            await asyncio.sleep(step)
            elapsed += step
            print(
                f"... {elapsed:>3}/{seconds}s | "
                f"messages captured: {len(live_messages)}"
            )

    except Exception as exc:
        print("\nWebSocket error:", type(exc).__name__, "-", exc)

    finally:
        if listener is not None:
            listener.cancel()
            await asyncio.gather(listener, return_exceptions=True)

        try:
            await ws.close()
        except Exception:
            pass

    print("\n" + "=" * 72)
    print(f"Captured {len(live_messages)} live price messages.")

    if not live_messages:
        print(
            "No price messages arrived. If the selected security is quiet or "
            "outside an active session, try the 24/7 BTC-USD test cell below."
        )

    return list(live_messages)

### Start the stream

**This cell is the part that was missing before.**  
Run it to actually start receiving data.

In [10]:
messages = await stream_yahoo_live(
    symbols=[TICKER],
    seconds=30,
)

Connecting to Yahoo WebSocket...
Symbols: ['SPY']
Listening for 30 seconds...

Connected to WebSocket.
Subscribed to symbols: ['SPY']
Listening for messages...
LIVE | SPY          price=    766.000000 time=2026-08-21 21:40:00+00:00 2
...   5/30s | messages captured: 1
LIVE | SPY          price=    766.006000 time=2026-08-21 21:40:01+00:00 2
...  10/30s | messages captured: 2
LIVE | SPY          price=    766.000000 time=2026-08-21 21:40:07+00:00 2
Heartbeat subscription sent for symbols: {'SPY'}
...  15/30s | messages captured: 3
LIVE | SPY          price=    766.000000 time=2026-08-21 21:40:11+00:00 2
...  20/30s | messages captured: 4
LIVE | SPY          price=    766.016700 time=2026-08-21 21:40:14+00:00 2
...  25/30s | messages captured: 5
LIVE | SPY          price=    766.010000 time=2026-08-21 21:40:24+00:00 2
Heartbeat subscription sent for symbols: {'SPY'}
...  30/30s | messages captured: 6
WebSocket listening interrupted. Closing connection...
WebSocket connection closed.
WebS

### 24/7 WebSocket diagnostic

If a stock/ETF stream is silent, run this separate diagnostic. Bitcoin trades continuously, so it is a useful test of whether the Yahoo WebSocket connection itself is working.

In [11]:
# Uncomment and run if the main stream is silent:
# btc_messages = await stream_yahoo_live(
#     symbols=["BTC-USD"],
#     seconds=30,
# )

## 8. Vectorized signal engine

The interpreter above explains **one current bar**.

For backtesting, we need the same general idea calculated across the full time series without using future information.

Signal meanings:

- `+1` = BUY / long
- `0` = NEUTRAL / flat
- `-1` = SELL / short signal

By default `ALLOW_SHORTS=False`, so bearish signals cause the backtest to go flat instead of opening a short position.

In [12]:
def add_rule_signal(df):
    out = df.copy()
    score = pd.Series(0.0, index=out.index)

    # Trend
    score += np.where(out["Close"] > out["SMA_50"], 1.0, -1.0)
    score += np.where(out["Close"] > out["SMA_200"], 1.0, -1.0)
    score += np.where(out["SMA_50"] > out["SMA_200"], 0.75, -0.75)
    score += np.where(out["EMA_9"] > out["SMA_50"], 0.50, -0.50)

    # RSI: momentum, but damp the vote when very stretched.
    rsi_vote = np.select(
        [
            out["RSI_14"] >= 70,
            out["RSI_14"] >= 55,
            out["RSI_14"] > 45,
            out["RSI_14"] > 30,
            out["RSI_14"] <= 30,
        ],
        [0.35, 1.0, 0.0, -1.0, -0.35],
        default=0.0,
    )
    score += rsi_vote

    # MACD
    score += np.where(out["MACD"] > out["MACD_SIGNAL"], 0.65, -0.65)
    score += np.where(out["MACD"] > 0, 0.35, -0.35)

    # Stochastic cross
    score += np.where(out["STOCH_K"] > out["STOCH_D"], 0.60, -0.60)

    out["RULE_SCORE"] = score

    signal = np.select(
        [
            out["RULE_SCORE"] >= BUY_THRESHOLD,
            out["RULE_SCORE"] <= SELL_THRESHOLD,
        ],
        [1, -1],
        default=0,
    )

    out["RAW_SIGNAL"] = signal.astype(int)

    # Do not generate signals until enough history exists for SMA200.
    out.loc[out["SMA_200"].isna(), "RAW_SIGNAL"] = 0

    out["SIGNAL_LABEL"] = out["RAW_SIGNAL"].map(
        {1: "BUY", 0: "NEUTRAL", -1: "SELL"}
    )

    return out


signal_df = add_rule_signal(df)

signal_df[
    ["Datetime", "Close", "RULE_SCORE", "RAW_SIGNAL", "SIGNAL_LABEL"]
].tail(20)

,Datetime,Close,RULE_SCORE,RAW_SIGNAL,SIGNAL_LABEL
1930,2026-08-21 15:40:00-04:00,766.169922,3.65,1,BUY
1931,2026-08-21 15:41:00-04:00,766.109985,2.35,1,BUY
1932,2026-08-21 15:42:00-04:00,766.224976,4.55,1,BUY
1933,2026-08-21 15:43:00-04:00,766.320007,5.85,1,BUY
1934,2026-08-21 15:44:00-04:00,766.469971,5.85,1,BUY
1935,2026-08-21 15:45:00-04:00,766.460022,4.65,1,BUY
1936,2026-08-21 15:46:00-04:00,766.500000,4.65,1,BUY
1937,2026-08-21 15:47:00-04:00,766.690002,5.20,1,BUY
1938,2026-08-21 15:48:00-04:00,766.659973,4.65,1,BUY
1939,2026-08-21 15:49:00-04:00,766.669983,4.65,1,BUY


## 9. No-look-ahead backtest + transaction costs

Important implementation detail:

The signal calculated from bar **t** is applied to the return beginning **after** that signal.  
That one-bar shift prevents us from earning a return before the signal existed.

Transaction costs are charged whenever the position changes.

In [13]:
def backtest_signals(
    signal_df,
    allow_shorts=ALLOW_SHORTS,
    transaction_cost_bps=TRANSACTION_COST_BPS,
):
    bt = signal_df.copy()

    if allow_shorts:
        target_position = bt["RAW_SIGNAL"].astype(float)
    else:
        target_position = (bt["RAW_SIGNAL"] == 1).astype(float)

    # Signal known at bar t becomes the position for the next bar.
    bt["POSITION"] = target_position.shift(1).fillna(0.0)

    bt["ASSET_RETURN"] = bt["Close"].pct_change().fillna(0.0)

    # One unit of turnover = move 0->1 or 1->0.
    # Reversing -1->+1 costs two units.
    bt["TURNOVER"] = bt["POSITION"].diff().abs().fillna(bt["POSITION"].abs())

    cost_rate = transaction_cost_bps / 10_000
    bt["COST"] = bt["TURNOVER"] * cost_rate

    bt["STRATEGY_RETURN_GROSS"] = bt["POSITION"] * bt["ASSET_RETURN"]
    bt["STRATEGY_RETURN"] = bt["STRATEGY_RETURN_GROSS"] - bt["COST"]

    bt["STRATEGY_EQUITY"] = (1 + bt["STRATEGY_RETURN"]).cumprod()
    bt["BUY_HOLD_EQUITY"] = (1 + bt["ASSET_RETURN"]).cumprod()

    bt["DRAWDOWN"] = (
        bt["STRATEGY_EQUITY"]
        / bt["STRATEGY_EQUITY"].cummax()
        - 1
    )

    return bt


bt = backtest_signals(signal_df)
bt.tail()

,Datetime,Open,High,Low,Close,Volume,EMA_9,EMA_20,SMA_50,SMA_100,...,SIGNAL_LABEL,POSITION,ASSET_RETURN,TURNOVER,COST,STRATEGY_RETURN_GROSS,STRATEGY_RETURN,STRATEGY_EQUITY,BUY_HOLD_EQUITY,DRAWDOWN
1945,2026-08-21 15:55:00-04:00,765.890015,765.909973,765.530029,765.849976,331280,766.116230,766.196986,766.152091,765.857473,...,SELL,0.0,-0.000065,0.0,0.0,-0.0,-0.0,0.966507,0.986107,-0.033493
1946,2026-08-21 15:56:00-04:00,765.844971,766.020020,765.770020,765.984985,176490,766.089981,766.176796,766.150791,765.863723,...,NEUTRAL,0.0,0.000176,0.0,0.0,0.0,0.0,0.966507,0.986281,-0.033493
1947,2026-08-21 15:57:00-04:00,765.989990,766.039978,765.869995,765.979980,383020,766.067981,766.158051,766.145791,765.869973,...,NEUTRAL,0.0,-0.000007,0.0,0.0,-0.0,-0.0,0.966507,0.986274,-0.033493
1948,2026-08-21 15:58:00-04:00,765.969971,766.169983,765.960022,766.104980,512201,766.075381,766.152997,766.143491,765.877573,...,NEUTRAL,0.0,0.000163,0.0,0.0,0.0,0.0,0.966507,0.986435,-0.033493
1949,2026-08-21 15:59:00-04:00,766.099976,766.234985,765.340027,765.650024,1235702,765.990310,766.105095,766.134492,765.879373,...,SELL,0.0,-0.000594,0.0,0.0,-0.0,-0.0,0.966507,0.985849,-0.033493


## 10. Strategy performance metrics

For very short samples, annualized metrics can be unstable. Total return and drawdown should always be read alongside the duration of the backtest.

In [14]:
def approximate_bars_per_year(interval):
    mapping = {
        "1m": 252 * 390,
        "2m": 252 * 195,
        "5m": 252 * 78,
        "15m": 252 * 26,
        "30m": 252 * 13,
        "60m": 252 * 6.5,
        "90m": 252 * (390 / 90),
        "1h": 252 * 6.5,
        "1d": 252,
        "5d": 52,
        "1wk": 52,
        "1mo": 12,
        "3mo": 4,
    }
    return mapping.get(interval, np.nan)


def performance_metrics(bt, interval=INTERVAL):
    r = bt["STRATEGY_RETURN"].dropna()
    active = bt.loc[bt["POSITION"] != 0, "STRATEGY_RETURN"].dropna()

    total_return = bt["STRATEGY_EQUITY"].iloc[-1] - 1
    buy_hold_return = bt["BUY_HOLD_EQUITY"].iloc[-1] - 1
    max_drawdown = bt["DRAWDOWN"].min()

    bpy = approximate_bars_per_year(interval)

    if len(r) > 1 and r.std(ddof=1) > 0 and pd.notna(bpy):
        sharpe = np.sqrt(bpy) * r.mean() / r.std(ddof=1)
    else:
        sharpe = np.nan

    positive = active[active > 0].sum()
    negative = active[active < 0].sum()

    profit_factor = (
        positive / abs(negative)
        if negative < 0
        else np.inf if positive > 0 else np.nan
    )

    win_rate = (active > 0).mean() if len(active) else np.nan

    entries = (
        (bt["POSITION"] != 0)
        & (bt["POSITION"].shift(1).fillna(0) == 0)
    ).sum()

    elapsed = bt["Datetime"].iloc[-1] - bt["Datetime"].iloc[0]

    return pd.DataFrame(
        {
            "metric": [
                "Sample duration",
                "Strategy total return",
                "Buy & hold total return",
                "Max drawdown",
                "Approx. annualized Sharpe",
                "Active-bar win rate",
                "Profit factor",
                "Entries",
                "Total turnover",
                "Transaction cost assumption",
            ],
            "value": [
                str(elapsed),
                f"{100 * total_return:.2f}%",
                f"{100 * buy_hold_return:.2f}%",
                f"{100 * max_drawdown:.2f}%",
                None if pd.isna(sharpe) else round(float(sharpe), 3),
                None if pd.isna(win_rate) else f"{100 * win_rate:.1f}%",
                None if pd.isna(profit_factor) else round(float(profit_factor), 3),
                int(entries),
                round(float(bt["TURNOVER"].sum()), 2),
                f"{TRANSACTION_COST_BPS:.2f} bps / unit turnover",
            ],
        }
    )


metrics = performance_metrics(bt)
metrics

,metric,value
0,Sample duration,4 days 06:29:00
1,Strategy total return,-3.35%
2,Buy & hold total return,-1.42%
3,Max drawdown,-3.35%
4,Approx. annualized Sharpe,-46.938
5,Active-bar win rate,39.7%
6,Profit factor,0.597
7,Entries,76
8,Total turnover,152.0
9,Transaction cost assumption,2.00 bps / unit turnover


In [15]:
def plot_backtest(bt, ticker=None):
    ticker = TICKER if ticker is None else ticker

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=bt["Datetime"],
            y=bt["STRATEGY_EQUITY"],
            name="Rule strategy",
        )
    )
    fig.add_trace(
        go.Scatter(
            x=bt["Datetime"],
            y=bt["BUY_HOLD_EQUITY"],
            name=f"{ticker} buy & hold",
        )
    )

    fig.update_layout(
        title=f"{ticker} — Backtest equity curve",
        xaxis_title="Time",
        yaxis_title="Growth of 1.0",
        hovermode="x unified",
        height=550,
    )

    fig.show()


plot_backtest(bt)

## 11. External benchmark comparison

This compares the strategy with a separate Yahoo benchmark over the same approximate period/interval.

If `TICKER == BENCHMARK_TICKER`, the external benchmark is naturally the same instrument.

In [16]:
def benchmark_comparison(bt, benchmark_ticker=BENCHMARK_TICKER):
    bench_raw = fetch_yahoo_data(
        ticker=benchmark_ticker,
        period=PERIOD,
        interval=INTERVAL,
        prepost=PREPOST,
    )

    bench = bench_raw[["Datetime", "Close"]].rename(
        columns={"Close": "Benchmark_Close"}
    )

    merged = pd.merge_asof(
        bt.sort_values("Datetime"),
        bench.sort_values("Datetime"),
        on="Datetime",
        direction="nearest",
        tolerance=pd.Timedelta("5min") if INTERVAL.endswith("m") else None,
    )

    merged["BENCHMARK_RETURN"] = merged["Benchmark_Close"].pct_change().fillna(0)
    merged["BENCHMARK_EQUITY"] = (1 + merged["BENCHMARK_RETURN"]).cumprod()

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=merged["Datetime"],
            y=merged["STRATEGY_EQUITY"],
            name=f"{TICKER} rule strategy",
        )
    )
    fig.add_trace(
        go.Scatter(
            x=merged["Datetime"],
            y=merged["BUY_HOLD_EQUITY"],
            name=f"{TICKER} buy & hold",
        )
    )
    fig.add_trace(
        go.Scatter(
            x=merged["Datetime"],
            y=merged["BENCHMARK_EQUITY"],
            name=f"{benchmark_ticker} benchmark",
        )
    )

    fig.update_layout(
        title="Strategy vs asset vs external benchmark",
        xaxis_title="Time",
        yaxis_title="Growth of 1.0",
        hovermode="x unified",
        height=550,
    )

    fig.show()
    return merged


benchmark_df = benchmark_comparison(bt)

## 12. Multi-ticker interpreter / scanner

This runs the same indicator interpreter across several symbols and produces a ranked table.

Keep the list modest because Yahoo can rate-limit excessive requests.

In [17]:
SCANNER_TICKERS = [
    "SPY",
    "QQQ",
    "AAPL",
    "MSFT",
    "NVDA",
    "AMZN",
    "META",
    "BTC-USD",
]


def scan_tickers(tickers=SCANNER_TICKERS):
    rows = []

    for symbol in tickers:
        try:
            x = add_indicators(
                fetch_yahoo_data(
                    ticker=symbol,
                    period=PERIOD,
                    interval=INTERVAL,
                    prepost=PREPOST,
                )
            )

            result = interpret_market_state(x, ticker=symbol)
            last = x.iloc[-1]

            rows.append(
                {
                    "ticker": symbol,
                    "state": result["label"],
                    "score": round(result["score"], 1),
                    "agreement_pct": (
                        np.nan
                        if pd.isna(result["rule_agreement_pct"])
                        else round(result["rule_agreement_pct"], 1)
                    ),
                    "close": round(float(last["Close"]), 4),
                    "rsi": round(float(last["RSI_14"]), 1)
                    if pd.notna(last["RSI_14"])
                    else np.nan,
                    "macd_hist": round(float(last["MACD_HIST"]), 6)
                    if pd.notna(last["MACD_HIST"])
                    else np.nan,
                    "atr_pct": round(float(last["ATR_PCT"]), 4)
                    if pd.notna(last["ATR_PCT"])
                    else np.nan,
                    "volume_ratio": round(float(last["VOLUME_RATIO"]), 2)
                    if pd.notna(last["VOLUME_RATIO"])
                    else np.nan,
                    "latest_bar": last["Datetime"],
                    "error": "",
                }
            )

        except Exception as exc:
            rows.append(
                {
                    "ticker": symbol,
                    "state": "ERROR",
                    "score": np.nan,
                    "agreement_pct": np.nan,
                    "close": np.nan,
                    "rsi": np.nan,
                    "macd_hist": np.nan,
                    "atr_pct": np.nan,
                    "volume_ratio": np.nan,
                    "latest_bar": pd.NaT,
                    "error": f"{type(exc).__name__}: {exc}",
                }
            )

    return (
        pd.DataFrame(rows)
        .sort_values("score", ascending=False, na_position="last")
        .reset_index(drop=True)
    )


scanner_results = scan_tickers()
scanner_results

,ticker,state,score,agreement_pct,close,rsi,macd_hist,atr_pct,volume_ratio,latest_bar,error
0,BTC-USD,STRONG BULLISH,57.3,71.4,78093.8984,55.8,-32.440024,0.1368,0.00,2026-08-21 21:40:00+00:00,
1,MSFT,NEUTRAL / MIXED,-11.3,57.1,483.3500,46.2,-0.062826,0.0718,4.08,2026-08-21 15:59:00-04:00,
2,QQQ,BEARISH,-43.5,75.0,713.4100,42.7,-0.044325,0.0383,3.25,2026-08-21 15:59:00-04:00,
3,AAPL,STRONG BEARISH,-72.6,87.5,309.4200,41.5,-0.051168,0.0827,4.84,2026-08-21 15:59:00-04:00,
4,SPY,STRONG BEARISH,-75.8,87.5,765.6500,34.4,-0.064638,0.0330,4.72,2026-08-21 15:59:00-04:00,
5,AMZN,STRONG BEARISH,-75.8,87.5,258.6500,35.1,-0.113993,0.0962,3.59,2026-08-21 15:59:00-04:00,
6,NVDA,STRONG BEARISH,-100.0,100.0,214.7500,31.2,-0.049811,0.0805,4.08,2026-08-21 15:59:00-04:00,
7,META,STRONG BEARISH,-100.0,100.0,549.9100,42.1,-0.087685,0.0742,4.76,2026-08-21 15:59:00-04:00,


## 13. Walk-forward ML model

This section asks a different question:

> Given only information available at bar **t**, can a simple model discriminate whether the close will be higher after `PREDICTION_HORIZON` bars?

Methodological safeguards included here:

- features use current/past observations only,
- future return is used only to construct the target,
- folds preserve time ordering,
- a `gap` equal to the forecast horizon separates train and test folds,
- scaling is fitted inside each training fold,
- predictions are genuinely out-of-sample for each fold.

The model is deliberately simple (`LogisticRegression`) so the pipeline is interpretable and leakage is easier to audit.

In [18]:
ML_FEATURES = [
    "RETURN_1",
    "RETURN_5",
    "RETURN_20",
    "RSI_14",
    "MACD_HIST",
    "STOCH_K",
    "STOCH_D",
    "ATR_PCT",
    "BB_POSITION",
    "BB_WIDTH_PCT",
    "VOLUME_RATIO",
]


def prepare_ml_data(df, horizon=PREDICTION_HORIZON):
    ml = df.copy()

    # Scale-sensitive price differences converted to relative features.
    ml["EMA9_DIST_PCT"] = 100 * (ml["Close"] / ml["EMA_9"] - 1)
    ml["SMA50_DIST_PCT"] = 100 * (ml["Close"] / ml["SMA_50"] - 1)
    ml["SMA200_DIST_PCT"] = 100 * (ml["Close"] / ml["SMA_200"] - 1)

    features = ML_FEATURES + [
        "EMA9_DIST_PCT",
        "SMA50_DIST_PCT",
        "SMA200_DIST_PCT",
    ]

    ml["FUTURE_RETURN"] = ml["Close"].shift(-horizon) / ml["Close"] - 1
    ml["TARGET_UP"] = (ml["FUTURE_RETURN"] > 0).astype(int)

    # Rows whose future target is genuinely unavailable must be removed,
    # otherwise the boolean comparison would turn NaN into class 0.
    ml.loc[ml["FUTURE_RETURN"].isna(), "TARGET_UP"] = np.nan

    ml = ml.dropna(subset=features + ["TARGET_UP"]).copy()
    ml["TARGET_UP"] = ml["TARGET_UP"].astype(int)

    return ml, features


ml_df, ml_features = prepare_ml_data(df)

print("ML rows:", len(ml_df))
print("Features:", len(ml_features))
print("Up-class proportion:", round(ml_df["TARGET_UP"].mean(), 3))

ML rows: 1746
Features: 14
Up-class proportion: 0.44


In [19]:
def walk_forward_logistic(
    ml_df,
    features,
    n_splits=5,
    horizon=PREDICTION_HORIZON,
):
    if len(ml_df) < 500:
        raise ValueError(
            "Not enough post-indicator rows for the default walk-forward analysis. "
            "Increase PERIOD or use a longer-history interval."
        )

    X = ml_df[features].astype(float)
    y = ml_df["TARGET_UP"].astype(int)

    splitter = TimeSeriesSplit(
        n_splits=n_splits,
        gap=horizon,
    )

    oof = pd.DataFrame(
        {
            "Datetime": ml_df["Datetime"],
            "y_true": y,
            "prob_up": np.nan,
            "fold": np.nan,
        },
        index=ml_df.index,
    )

    fold_rows = []

    for fold, (train_idx, test_idx) in enumerate(splitter.split(X), start=1):
        X_train = X.iloc[train_idx]
        y_train = y.iloc[train_idx]
        X_test = X.iloc[test_idx]
        y_test = y.iloc[test_idx]

        model = Pipeline(
            [
                ("scale", StandardScaler()),
                (
                    "model",
                    LogisticRegression(
                        max_iter=2000,
                        class_weight="balanced",
                    ),
                ),
            ]
        )

        model.fit(X_train, y_train)

        prob = model.predict_proba(X_test)[:, 1]
        pred = (prob >= 0.5).astype(int)

        oof.loc[X_test.index, "prob_up"] = prob
        oof.loc[X_test.index, "fold"] = fold

        auc = (
            roc_auc_score(y_test, prob)
            if y_test.nunique() == 2
            else np.nan
        )

        fold_rows.append(
            {
                "fold": fold,
                "train_n": len(train_idx),
                "test_n": len(test_idx),
                "auc": auc,
                "accuracy": accuracy_score(y_test, pred),
                "balanced_accuracy": balanced_accuracy_score(y_test, pred),
                "test_start": ml_df.iloc[test_idx]["Datetime"].iloc[0],
                "test_end": ml_df.iloc[test_idx]["Datetime"].iloc[-1],
            }
        )

    valid = oof["prob_up"].notna()
    y_valid = oof.loc[valid, "y_true"].astype(int)
    p_valid = oof.loc[valid, "prob_up"].astype(float)
    pred_valid = (p_valid >= 0.5).astype(int)

    overall_auc = (
        roc_auc_score(y_valid, p_valid)
        if y_valid.nunique() == 2
        else np.nan
    )

    summary = pd.DataFrame(
        {
            "metric": [
                "Out-of-sample rows",
                "Overall walk-forward AUC",
                "Overall accuracy",
                "Overall balanced accuracy",
                "Forecast horizon (bars)",
            ],
            "value": [
                int(valid.sum()),
                None if pd.isna(overall_auc) else round(float(overall_auc), 4),
                round(float(accuracy_score(y_valid, pred_valid)), 4),
                round(float(balanced_accuracy_score(y_valid, pred_valid)), 4),
                horizon,
            ],
        }
    )

    return oof, pd.DataFrame(fold_rows), summary


oof_predictions, fold_metrics, ml_summary = walk_forward_logistic(
    ml_df,
    ml_features,
)

display(ml_summary)
display(fold_metrics)

,metric,value
0,Out-of-sample rows,1455.0000
1,Overall walk-forward AUC,0.5453
2,Overall accuracy,0.5450
3,Overall balanced accuracy,0.5311
4,Forecast horizon (bars),5.0000


,fold,train_n,test_n,auc,accuracy,balanced_accuracy,test_start,test_end
0,1,286,291,0.586383,0.560137,0.518060,2026-08-18 11:10:00-04:00,2026-08-19 09:30:00-04:00
1,2,577,291,0.576208,0.618557,0.585698,2026-08-19 09:31:00-04:00,2026-08-19 14:21:00-04:00
2,3,868,291,0.599726,0.594502,0.598072,2026-08-19 14:22:00-04:00,2026-08-20 12:42:00-04:00
3,4,1159,291,0.485513,0.501718,0.487773,2026-08-20 12:43:00-04:00,2026-08-21 11:03:00-04:00
4,5,1450,291,0.465674,0.450172,0.444752,2026-08-21 11:04:00-04:00,2026-08-21 15:54:00-04:00


In [20]:
def plot_walk_forward_predictions(oof_predictions):
    data = oof_predictions.dropna(subset=["prob_up"]).copy()

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=data["Datetime"],
            y=data["prob_up"],
            name="Out-of-sample P(up)",
            mode="lines",
        )
    )

    fig.add_hline(y=0.5, line_dash="dash")

    fig.update_layout(
        title=(
            f"{TICKER} — Walk-forward probability of positive "
            f"{PREDICTION_HORIZON}-bar return"
        ),
        xaxis_title="Time",
        yaxis_title="Predicted probability",
        yaxis_range=[0, 1],
        hovermode="x unified",
        height=500,
    )

    fig.show()


plot_walk_forward_predictions(oof_predictions)

## 14. Daily / long-horizon mode

For conventional daily moving averages:

```python
TICKER = "SPY"
PERIOD = "2y"
INTERVAL = "1d"

raw_df = fetch_yahoo_data()
df = add_indicators(raw_df)

technical_dashboard(df, display_bars=None)

interpretation = interpret_market_state(df)
print_interpretation(interpretation)

signal_df = add_rule_signal(df)
bt = backtest_signals(signal_df)

display(performance_metrics(bt))
plot_backtest(bt)
```

Then SMA50, SMA100 and SMA200 correspond to approximately 50, 100 and 200 trading days.